In [ ]:
# Model Training Notebook
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, r2_score

from data.preprocessing import BiomassDataPreprocessor
from models.biomass_predictor import AdvancedBiomassPredictor

# Configuration
config = {
    'data_path': '../data/raw/training_data.csv',
    'images_dir': '../data/raw/images/',
    'model_save_path': '../trained_models/biomass_predictor',
    'test_size': 0.2,
    'val_size': 0.1,
    'epochs': 500,
    'batch_size': 32,
    'early_stopping_patience': 50,
    'learning_rate': 0.001
}

# Initialize preprocessor
preprocessor = BiomassDataPreprocessor(config)

# Load and validate data
print("Loading data...")
data = preprocessor.load_and_validate_data(config['data_path'])

# Prepare features and targets
print("Preparing features and targets...")
features, targets, valid_indices = preprocessor.prepare_training_data(
    data, config['images_dir']
)

# Split data
X_train, X_val, X_test, y_train, y_val, y_test = preprocessor.split_data(
    features, targets, 
    test_size=config['test_size'],
    val_size=config['val_size']
)

# Scale features
X_train_scaled, X_val_scaled, X_test_scaled = preprocessor.scale_features(
    X_train, X_val, X_test
)

# Scale targets
y_train_scaled, y_val_scaled, y_test_scaled = preprocessor.scale_targets(
    y_train, y_val, y_test
)

# Initialize and train model
model = AdvancedBiomassPredictor(config)

print("Training models...")
# Train Neural Network
nn_history = model.train_neural_network(
    X_train_scaled, y_train_scaled, X_val_scaled, y_val_scaled
)

# Train XGBoost
xgb_model = model.train_xgboost(X_train, y_train, X_val, y_val)

# Train Random Forest
rf_model = model.train_random_forest(X_train, y_train)

# Evaluate models
print("Evaluating models...")

# Neural Network evaluation
nn_metrics = model.evaluate_model(X_test_scaled, y_test_scaled, 'neural_network')
print("Neural Network Results:")
print(f"Competition Score: {nn_metrics['competition_score']:.4f}")

# XGBoost evaluation  
xgb_metrics = model.evaluate_model(X_test, y_test, 'xgboost')
print("XGBoost Results:")
print(f"Competition Score: {xgb_metrics['competition_score']:.4f}")

# Random Forest evaluation
rf_metrics = model.evaluate_model(X_test, y_test, 'random_forest')
print("Random Forest Results:")
print(f"Competition Score: {rf_metrics['competition_score']:.4f}")

# Ensemble evaluation
ensemble_pred, all_predictions = model.create_ensemble_prediction(
    X_test_scaled,
    weights={'neural_network': 0.6, 'xgboost': 0.3, 'random_forest': 0.1}
)

ensemble_competition_score, ensemble_component_scores = model.evaluate_competition_metric(
    y_test, ensemble_pred
)
print(f"Ensemble Competition Score: {ensemble_competition_score:.4f}")

# Save models and preprocessors
print("Saving models...")
model.save_model(config['model_save_path'], 'neural_network')
model.save_model(config['model_save_path'], 'xgboost') 
model.save_model(config['model_save_path'], 'random_forest')
preprocessor.save_preprocessors(config['model_save_path'])

# Plot training history
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(nn_history.history['loss'], label='Training Loss')
plt.plot(nn_history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(nn_history.history['Dry_Total_g_mae'], label='Total Biomass MAE')
plt.plot(nn_history.history['val_Dry_Total_g_mae'], label='Val Total Biomass MAE')
plt.title('Total Biomass MAE')
plt.ylabel('MAE')
plt.xlabel('Epoch')
plt.legend()

plt.tight_layout()
plt.savefig('../reports/training_history.png', dpi=300, bbox_inches='tight')
plt.show()

# Feature importance analysis (for tree-based models)
feature_importance = np.mean(
    [est.feature_importances_ for est in model.xgb_model.estimators_], 
    axis=0
)

plt.figure(figsize=(10, 6))
indices = np.argsort(feature_importance)[-20:]  # Top 20 features
plt.barh(range(len(indices)), feature_importance[indices])
plt.yticks(range(len(indices)), [f'Feature {i}' for i in indices])
plt.title('Top 20 Feature Importances (XGBoost)')
plt.tight_layout()
plt.savefig('../reports/feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("Model training completed successfully!")